In [ ]:
import logging
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch
import numpy as np
import pandas as pd
import json

logging.basicConfig(level=logging.INFO)

sys.path.insert(0, str(Path.cwd().parents[0]))
sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

# Generation Parameters

In [ ]:
NUM_SYNTH = 1
GEN_NAME = "simple_test"
SEED = 2
PARAMS_PATH = Path("../config/base_params.json")

SHOW_IMGS = False
SAVE_IMGS = True
SAVE_IMGS_DIR = Path("../segm/data/")

# Training Parameters

In [ ]:
REAL_IMGS_DIR = Path("../segm/real_data/labeled_wo_test")
TEST_IMGS_DIR = Path("../segm/real_data/test")

MODEL_NAME = "model_test"

RESULTS_DIR = Path("../segm/results/")

ALLOW_OVERWRITE_DATASET = False
ALLOW_OVERWRITE_RUN = False

# Image Generation

In [ ]:
from main import generate

generate(
    num=NUM_SYNTH,
    gen=GEN_NAME,
    img_params_path=str(PARAMS_PATH),
    img_save_path=str(SAVE_IMGS_DIR),
    show=SHOW_IMGS,
    save=SAVE_IMGS,
    seed=SEED,
)

# Load Images

In [ ]:
SYNTH_DIR = SAVE_IMGS_DIR  # can be changed to work with previously generated images
LOAD_SYNTH_MODE = "basic"  # "basic" or "clustered"

# only used for "basic"
LOAD_NUM_SYNTH = 5  

# only used for "clustered"
CLUSTERS_BASE_COUNTS = (10, 13, 7, 9, 6)  
CLUSTERS_MULTIPLICATOR = 5
CLUSTERS_DIR_PREFIX = "cl"  # assumes the clustered images were generated like *{prefix}{i}/img.png

In [ ]:
from precipseg import data

# test
test_img_paths, test_label_paths = data.load_paths(TEST_IMGS_DIR)
test_imgs = [data.read_image_normalized(p) for p in test_img_paths]
test_labels = [data.read_image_normalized(p) for p in test_label_paths]

# train real
real_img_paths, real_label_paths = data.load_paths(REAL_IMGS_DIR)

# train synthetic
if LOAD_SYNTH_MODE == "basic":
    synth_img_paths, synth_label_paths = data.load_paths(SYNTH_DIR, LOAD_NUM_SYNTH)
elif LOAD_SYNTH_MODE == "clustered":
    synth_img_paths, synth_label_paths = data.load_clustered(SYNTH_DIR, CLUSTERS_BASE_COUNTS, CLUSTERS_MULTIPLICATOR)
else:
    raise ValueError(f"Unknown LOAD_SYNTH_MODE={LOAD_SYNTH_MODE!r}")


train_img_paths = real_img_paths + synth_img_paths
train_label_paths = real_label_paths + synth_label_paths

print(f"Train: {len(real_img_paths)} real + {len(synth_img_paths)} synthetic ({LOAD_SYNTH_MODE}).")
print(f"Test: {len(test_img_paths)}.")

# Prepare Dataloaders

In [ ]:
from precipseg import data

PATCH_SIZE = 256
X_CHANNELS = 3
BATCH_SIZE = 16
VAL_SIZE = 0.25


train_dl, val_dl = data.build_dataloaders(
    train_img_paths, train_label_paths,
    patch_size=PATCH_SIZE,
    x_channels=X_CHANNELS,
    batch_size=BATCH_SIZE,
    val_size=VAL_SIZE,
)

In [ ]:
from precipseg.data import show_augmented_triple

show_augmented_triple(n_show=5, train_dl=train_dl)


# Model

In [ ]:
from segmentation_models_pytorch import Unet

model = Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=X_CHANNELS,
    classes=1,
    activation=None,
)

# Loss Selection

In [ ]:
from precipseg.losses import make_loss, available_losses

print("Available losses:", available_losses())

LOSS_NAME = "focal_dice"
loss_fn = make_loss(LOSS_NAME)
# loss_fn = make_loss("focal_dice", gamma=2.0, alpha=0.25)
print("Chosen loss:", LOSS_NAME)

# Training

In [ ]:
from precipseg.training import train

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

models_dir = Path("../segm/models")
models_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = models_dir / f"checkpoint-{MODEL_NAME}"

log_path = RESULTS_DIR / MODEL_NAME / "log.txt"

In [ ]:
from tqdm.contrib.logging import logging_redirect_tqdm

with logging_redirect_tqdm():
    train_loss, val_loss = train(
        model,
        train_dl,
        val_dl,
        loss_fn,
        checkpoint_path=checkpoint_path,
        patience=10,
        scheduler_patience=5,
        device=DEVICE,
        log_path=log_path,
        real_imgs_num=len(real_img_paths),
        synth_imgs_num=len(synth_img_paths),
        synth_path=str(SYNTH_DIR/GEN_NAME),
    )

torch.save(model.state_dict(), models_dir / f"{MODEL_NAME}.pt")

# Predictions

In [ ]:
from precipseg import predictions 


test_preds = predictions.segment_many(model, test_imgs, device=DEVICE, x_channels=X_CHANNELS)

In [ ]:
for img, label, pred in zip(test_imgs, test_labels, test_preds):
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(img,vmin=0, vmax=1, cmap="gray")
    axs[1].imshow(label,vmin=0, vmax=1, cmap="gray")
    axs[2].imshow(pred,vmin=0, vmax=1, cmap="gray")

    plt.suptitle("Img, Mask, Prediction")

    H, W = img.shape
    crop_w, crop_h = 250, 250

    x0 = np.random.randint(0, W - crop_w)
    y0 = np.random.randint(0, H - crop_h)
    x1 = x0 + crop_w
    y1 = y0 + crop_h

    for ax in axs:
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
    plt.show()

In [ ]:
# train and val loss curves

plt.figure(figsize=(6,4))
plt.plot(train_loss, label="train")
plt.plot(val_loss, label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training curve")
plt.legend()
plt.show()

# Save Results

In [ ]:
from precipseg.data import save_image_results


results_dir = Path("../segm/results") / MODEL_NAME
results_dir.mkdir(parents=True, exist_ok=True)

save_image_results(results_dir, test_imgs, test_labels, test_preds)

In [ ]:
from precipseg.metrics import compute_metric_curves, save_metric_curves

THRESHOLDS = np.linspace(0.10, 0.95, 10)
dice_vals, iou_vals, mean_p, mean_r, mean_f1 = compute_metric_curves(test_labels, test_preds, THRESHOLDS)
save_metric_curves(results_dir / "metrics.txt", THRESHOLDS, dice_vals, iou_vals, mean_p, mean_r, mean_f1)

In [ ]:
def print_best(metric_name, thresholds, dice, iou, precision, recall, f1):
    metric_map = {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    values = metric_map[metric_name]
    idx = np.nanargmax(values)
    print(f"Best {metric_name.upper()}: {values[idx]:.4f} at threshold {thresholds[idx]:.2f}")


print_best("dice", THRESHOLDS, dice_vals, iou_vals, mean_p, mean_r, mean_f1)